<a href="https://colab.research.google.com/github/berkemremert/eva_dialog_trial_collab/blob/main/eva_dialog_trial_collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ultravox v0.5 (Llama 3.2 1B) — Colab setup

The notebook is split so expensive work is not repeated:

1. Run **Environment**, **Configuration**, and **Hugging Face login** once per Colab session.
2. Leave cache cleanup disabled unless remote model code is broken or stale.
3. Run **Load model** once. Re-running that cell skips loading while `model` is already in memory.
4. Run the lightweight memory/status cell whenever needed.

> This T4-friendly checkpoint uses a 1B language-model backbone. The first download takes a few minutes; later loads are faster while the Colab cache survives.

## 0. Install compatible dependencies

This Ultravox v0.5 checkpoint declares Transformers 4.48.1. Run this before importing Transformers so its remote model code uses the matching library version.

> If Transformers was already imported in this Colab session, run this cell, choose **Runtime → Restart session**, and then run the notebook from the top. The downloaded model weights remain cached.

In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "transformers==4.48.1",
        "accelerate==1.2.1",
        "peft==0.14.0",
        "librosa>=0.11.0",
    ]
)
print("✅ Compatible dependencies installed")

## 1. Environment and imports

Run once after connecting to a GPU runtime.

In [ ]:
import gc
import os
import shutil

import huggingface_hub
import torch
import transformers
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModel

EXPECTED_TRANSFORMERS_VERSION = "4.48.1"
if transformers.__version__ != EXPECTED_TRANSFORMERS_VERSION:
    raise RuntimeError(
        f"Transformers {transformers.__version__} is active, but Ultravox needs "
        f"{EXPECTED_TRANSFORMERS_VERSION}. Run cell 0, restart the Colab session, "
        "and run the notebook from the top."
    )

print("Transformers:", transformers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(properties.total_memory / 1024**3, 1), "GB")
else:
    print("⚠️ In Colab, select Runtime → Change runtime type → GPU.")

## 2. Configuration

Keep both switches `False` for normal use.

In [ ]:
MODEL_ID = "fixie-ai/ultravox-v0_5-llama-3_2-1b"
CLEAR_REMOTE_CODE_CACHE = False
FORCE_RELOAD = False

print("Model:", MODEL_ID)
print("Clear remote-code cache:", CLEAR_REMOTE_CODE_CACHE)
print("Force model reload:", FORCE_RELOAD)

## 3. Hugging Face login

Add `HF_TOKEN` under Colab's **Secrets** panel before running this cell. The token's account must have accepted access to [Llama 3.2 1B Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct).

In [ ]:
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Add HF_TOKEN to Colab Secrets and allow this notebook to access it."
    ) from exc

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is empty. Update it in Colab Secrets.")

login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Hugging Face login successful")

## 4. Optional remote-code cache cleanup

This is normally skipped. Enable `CLEAR_REMOTE_CODE_CACHE` in Configuration only when a stale remote-code error needs fixing. It does not delete the downloaded model weights.

In [ ]:
REMOTE_CODE_CACHE = os.path.expanduser(
    "~/.cache/huggingface/modules/transformers_modules"
)

if CLEAR_REMOTE_CODE_CACHE:
    if os.path.exists(REMOTE_CODE_CACHE):
        shutil.rmtree(REMOTE_CODE_CACHE)
        print("✅ Transformers remote-code cache cleared")
    else:
        print("Remote-code cache is already empty")
else:
    print("⏭️ Cache cleanup skipped")

## 5. Load Ultravox v0.5 1B (run once)

This is the expensive block. If the model is already loaded, re-running the cell skips it. Set `FORCE_RELOAD = True` only when you intentionally want to discard and reload the in-memory model.

In [ ]:
model_is_loaded = "model" in globals() and model is not None

if model_is_loaded and not FORCE_RELOAD:
    print("✅ Model is already loaded; skipping reload")
else:
    if model_is_loaded:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("Loading:", MODEL_ID)
    model = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        device_map="auto",
        torch_dtype=torch.float16,
        token=HF_TOKEN,
    )
    model.eval()
    print("✅ Ultravox v0.5 Llama 3.2 1B loaded successfully")

## 6. Model and GPU status

This block is quick and safe to rerun.

In [ ]:
print("Model loaded:", "model" in globals() and model is not None)

if torch.cuda.is_available():
    print("GPU allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
    print("GPU reserved:", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

## 7. Prepare voice conversation (run once)

This reuses the model already in GPU memory. Ultravox listens and produces text; the browser's speech synthesizer reads the reply aloud.

In [ ]:
from transformers import AutoProcessor

if "model" not in globals() or model is None:
    raise RuntimeError("Run the model-loading cell first.")

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
)

def generate_ultravox(
    turns, audio=None, sampling_rate=16000, max_new_tokens=128, min_new_tokens=0
):
    prompt_turns = [dict(turn) for turn in turns]
    if audio is not None:
        prompt_turns.append({"role": "user", "content": "<|audio|>"})

    prompt = processor.tokenizer.apply_chat_template(
        prompt_turns,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = processor(
        text=prompt,
        audio=audio,
        sampling_rate=sampling_rate,
        return_tensors="pt",
    )

    input_device = next(model.parameters()).device
    inputs = inputs.to(input_device)
    if "audio_values" in inputs:
        inputs["audio_values"] = inputs["audio_values"].to(model.dtype)

    input_length = inputs["input_ids"].shape[1]
    terminators = [processor.tokenizer.eos_token_id]
    if "<|eot_id|>" in processor.tokenizer.get_vocab():
        terminators.append(processor.tokenizer.convert_tokens_to_ids("<|eot_id|>"))

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            do_sample=False,
            min_new_tokens=min_new_tokens,
            max_new_tokens=max_new_tokens,
            repetition_penalty=1.05,
            eos_token_id=terminators,
        )

    return processor.decode(
        output_ids[0, input_length:],
        skip_special_tokens=True,
    ).strip()

print("✅ Conversation processor ready")

## 8. Enable the Colab microphone (run once)

Your browser will ask for microphone permission the first time.

In [ ]:
import base64
import json
import subprocess
import time

import librosa
from google.colab import output
from IPython.display import Audio, Javascript, display

RECORDING_JAVASCRIPT = r"""
async function recordUntilStopped() {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const preferred = 'audio/webm;codecs=opus';
  const options = MediaRecorder.isTypeSupported(preferred) ? {mimeType: preferred} : {};
  const recorder = new MediaRecorder(stream, options);
  const chunks = [];
  recorder.ondataavailable = event => { if (event.data.size > 0) chunks.push(event.data); };

  const panel = document.createElement('div');
  panel.style = 'padding:12px;margin:8px 0;border:1px solid #ccc;border-radius:8px;';
  const status = document.createElement('span');
  status.textContent = '🔴 Recording… ';
  const stopButton = document.createElement('button');
  stopButton.textContent = 'Stop recording';
  panel.appendChild(status);
  panel.appendChild(stopButton);
  document.body.appendChild(panel);

  recorder.start();
  await new Promise(resolve => stopButton.onclick = resolve);
  const stopped = new Promise(resolve => recorder.addEventListener('stop', resolve, {once: true}));
  recorder.stop();
  await stopped;
  stream.getTracks().forEach(track => track.stop());
  panel.remove();

  const blob = new Blob(chunks, {type: recorder.mimeType || 'audio/webm'});
  return await new Promise(resolve => {
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  });
}
"""

def record_from_microphone():
    display(Javascript(RECORDING_JAVASCRIPT))
    data_url = output.eval_js("recordUntilStopped()")
    encoded_audio = data_url.split(",", 1)[1]
    timestamp = int(time.time() * 1000)
    webm_path = f"/content/ultravox_turn_{timestamp}.webm"
    wav_path = f"/content/ultravox_turn_{timestamp}.wav"

    with open(webm_path, "wb") as audio_file:
        audio_file.write(base64.b64decode(encoded_audio))

    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", webm_path, "-ac", "1", "-ar", "16000", wav_path,
        ],
        check=True,
    )
    return wav_path

def speak_in_browser(text):
    safe_text = json.dumps(text)
    display(Javascript(f"""
        window.speechSynthesis.cancel();
        const utterance = new SpeechSynthesisUtterance({safe_text});
        utterance.lang = 'tr-TR';
        const turkishVoice = window.speechSynthesis
          .getVoices()
          .find(voice => voice.lang.toLowerCase().startsWith('tr'));
        if (turkishVoice) utterance.voice = turkishVoice;
        window.speechSynthesis.speak(utterance);
    """))

print("✅ Microphone helpers ready")

## 9. Start or reset the conversation

Run this once at the beginning, or again whenever you want to erase the conversation history.

In [ ]:
SYSTEM_PROMPT = (
    "Sen Türkçe konuşan, dikkatli ve yardımsever bir sesli asistansın. "
    "Kullanıcı başka bir dilde konuşsa bile her zaman doğal ve konuyla ilgili Türkçe cevap ver. "
    "Mesaj belirsizse tahmin yürütme; ne demek istediğini anlamak için kısa bir "
    "açıklayıcı soru sor. İlgisiz tek kelimelik cevaplar verme. Önceki konuşmayı hatırla."
)
conversation_messages = []
MAX_HISTORY_MESSAGES = 12
print("✅ New conversation started")

## 10. Talk for one turn (rerun this cell)

Run the cell, speak, then press **Stop recording**. Rerun it for every new turn. The first turn can be slower because components are warming up.

In [ ]:
wav_path = record_from_microphone()
display(Audio(wav_path, autoplay=False))
audio, sample_rate = librosa.load(wav_path, sr=16000, mono=True)

transcript = generate_ultravox(
    [
        {
            "role": "system",
            "content": (
                "Kullanıcının konuşmasını duyulduğu dilde doğru biçimde yazıya dök. "
                "Yalnızca transkripti ver; tırnak veya açıklama ekleme."
            ),
        }
    ],
    audio=audio,
    sampling_rate=sample_rate,
    max_new_tokens=96,
)
print("You:", transcript)

recent_messages = conversation_messages[-MAX_HISTORY_MESSAGES:]
reply = generate_ultravox(
    [{"role": "system", "content": SYSTEM_PROMPT}] + recent_messages,
    audio=audio,
    sampling_rate=sample_rate,
    min_new_tokens=6,
    max_new_tokens=128,
)
conversation_messages.append({"role": "user", "content": transcript})
conversation_messages.append({"role": "assistant", "content": reply})

print("Ultravox:", reply)
speak_in_browser(reply)